# 🔬 WACV 2019 PCB Component Dataset Cleaning & SAM Point Extractor

This notebook processes the **WACV 2019 PCB Component Detection Dataset** (47 boards, 18,201 components):
1. Parses Pascal VOC XML annotations (`<object><name>ic...</name><bndbox>...`)
2. Filters out silkscreen markings (`text`, `pads`)
3. Cleans component bounding boxes (`ic`, `connector`, `led`, `clock`, `capacitor`, `diode`, `resistor`)
4. Prompts **Meta SAM ViT-B** to extract pixel-exact component masks and polygon vertices `[(x, y), ...]`
5. Maps to **KiCad Library Convention (KLC)** Reference Designators (`U1`, `C1`, `R1`, `J1`) and footprints
6. Exports **Excel (`.xlsx`)**, **CSV (`.csv`)**, and **LabelMe (`.json`)** format

In [ ]:
# 1. Install & Import Dependencies
!pip install -q opencv-python numpy pandas openpyxl matplotlib
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

import os
import json
import shutil
import xml.etree.ElementTree as ET
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from segment_anything import sam_model_registry, SamPredictor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using compute device: {device}')

In [ ]:
# 2. Download WACV 2019 Dataset (if not already downloaded)
import urllib.request
import zipfile

WACV_DIR = Path('wacv_data/pcb_wacv_2019')
ZIP_PATH = Path('wacv_data/pcb_wacv_2019.zip')
WACV_URL = 'https://ripl.cc.gatech.edu/data/pcb_wacv_2019.zip'

if not WACV_DIR.exists():
    print('Downloading WACV 2019 dataset from Georgia Tech RIPL (284 MB)...')
    WACV_DIR.parent.mkdir(parents=True, exist_ok=True)
    if not ZIP_PATH.exists():
        urllib.request.urlretrieve(WACV_URL, str(ZIP_PATH))
    print('Extracting...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall('wacv_data')
    print('WACV extraction complete!')
else:
    print('WACV 2019 dataset already present.')

In [ ]:
# 3. KiCad Library Convention (KLC) Class Mapping Table
WACV_CLASSES = {
    'capacitor': {'kicad': 'Capacitor_SMD', 'footprint': 'Capacitor_SMD:C_0805_2012Metric', 'ref_prefix': 'C'},
    'electrolytic': {'kicad': 'Capacitor_THT', 'footprint': 'Capacitor_THT:CP_Radial_D6.3mm_P2.50mm', 'ref_prefix': 'C'},
    'resistor': {'kicad': 'Resistor_SMD', 'footprint': 'Resistor_SMD:R_0805_2012Metric', 'ref_prefix': 'R'},
    'ic': {'kicad': 'Package_SO', 'footprint': 'Package_SO:SOIC-8_3.9x4.9mm_P1.27mm', 'ref_prefix': 'U'},
    'transistor': {'kicad': 'Package_TO_SOT_SMD', 'footprint': 'Package_TO_SOT_SMD:SOT-23', 'ref_prefix': 'Q'},
    'diode': {'kicad': 'Diode_SMD', 'footprint': 'Diode_SMD:D_SOD-123', 'ref_prefix': 'D'},
    'connector': {'kicad': 'Connector', 'footprint': 'Connector_PinHeader_2.54mm:PinHeader_1x04_P2.54mm_Vert', 'ref_prefix': 'J'},
    'inductor': {'kicad': 'Inductor_SMD', 'footprint': 'Inductor_SMD:L_0805_2012Metric', 'ref_prefix': 'L'},
    'switch': {'kicad': 'Button_Switch_SMD', 'footprint': 'Button_Switch_SMD:SW_Push_SPST_NO_Alps_SKRK', 'ref_prefix': 'SW'},
    'button': {'kicad': 'Button_Switch_SMD', 'footprint': 'Button_Switch_SMD:SW_Push_SPST_NO_Alps_SKRK', 'ref_prefix': 'SW'},
    'led': {'kicad': 'LED_SMD', 'footprint': 'LED_SMD:LED_0805_2012Metric', 'ref_prefix': 'D'},
    'clock': {'kicad': 'Crystal', 'footprint': 'Crystal:Crystal_SMD_3225-4Pin_3.2x2.5mm', 'ref_prefix': 'Y'},
    'fuse': {'kicad': 'Fuse', 'footprint': 'Fuse:Fuse_1206_3216Metric', 'ref_prefix': 'F'},
    'transformer': {'kicad': 'Transformer_SMD', 'footprint': 'Transformer_SMD:Transformer_Bourns_SRF0703', 'ref_prefix': 'T'}
}
IGNORED_LABELS = {'text', 'pads', 'pins', 'unknown', 'test'}

In [ ]:
# 4. Process Board XML, Run SAM & Extract Polygon Points
board_folder = Path('wacv_data/pcb_wacv_2019/ArduinoMega_Top')
xml_path = list(board_folder.glob('*.xml'))[0]
img_path = list(board_folder.glob('*.jpg'))[0]
output_dir = Path('wacv_sam_output')
output_dir.mkdir(parents=True, exist_ok=True)

img = cv2.imread(str(img_path))
h, w = img.shape[:2]

# Load SAM
sam = sam_model_registry['vit_b'](checkpoint='sam_vit_b.pth')
sam.to(device=device)
sam.eval()
predictor = SamPredictor(sam)
predictor.set_image(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

# Parse XML objects
tree = ET.parse(xml_path)
root = tree.getroot()
records = []
shapes = []
ref_counts = {}
vis_img = img.copy()

for idx, obj in enumerate(root.findall('object'), 1):
    raw_name = obj.find('name').text if obj.find('name') is not None else ''
    first_word = raw_name.strip().strip('"').lower().split()[0] if raw_name else ''
    if first_word in IGNORED_LABELS:
        continue
    bnd = obj.find('bndbox')
    if bnd is None: continue
    x1, y1 = float(bnd.find('xmin').text), float(bnd.find('ymin').text)
    x2, y2 = float(bnd.find('xmax').text), float(bnd.find('ymax').text)
    
    # Prompt SAM
    box_arr = np.array([x1, y1, x2, y2])
    masks, scores, _ = predictor.predict(box=box_arr[None, :], multimask_output=False)
    conf = float(scores[0])
    
    # Extract Contour
    contours, _ = cv2.findContours(masks[0].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: continue
    cnt = max(contours, key=cv2.contourArea)
    approx = cv2.approxPolyDP(cnt, 0.005 * cv2.arcLength(cnt, True), True)
    pts = [[round(float(p[0][0]), 2), round(float(p[0][1]), 2)] for p in approx]
    
    info = WACV_CLASSES.get(first_word, {'kicad': 'Generic_Component', 'footprint': 'Package_SO:SOIC-8', 'ref_prefix': 'U'})
    pfx = info['ref_prefix']
    ref_counts[pfx] = ref_counts.get(pfx, 0) + 1
    ref_des = f'{pfx}{ref_counts[pfx]}'
    
    records.append({
        'board': board_folder.name,
        'ref_des': ref_des,
        'type': first_word,
        'kicad_footprint': info['footprint'],
        'confidence': round(conf, 3),
        'num_points': len(pts),
        'points_compact': '; '.join([f'({p[0]},{p[1]})' for p in pts]),
        'points_json': json.dumps(pts)
    })
    
    cnt_pts = np.array(pts, dtype=np.int32).reshape((-1, 1, 2))
    cv2.polylines(vis_img, [cnt_pts], True, (0, 255, 0), 2)

df = pd.DataFrame(records)
df.to_excel(output_dir / f'{board_folder.name}_sam_points.xlsx', index=False)
df.to_csv(output_dir / f'{board_folder.name}_sam_points.csv', index=False)
print(f'Processed {len(records)} components on {board_folder.name}!')
df.head(10)

In [ ]:
# 5. Display Processed PCB Image with Polygons
plt.figure(figsize=(14, 10))
plt.imshow(cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB))
plt.title(f'WACV SAM Segmentation: {board_folder.name}')
plt.axis('off')
plt.show()